# Heat Equation — Physics-Informed Neural Network

The **heat equation** (also called the diffusion equation) is one of the fundamental
PDEs of mathematical physics. It models how a quantity — temperature, chemical
concentration, probability — spreads through a medium over time.

## The Equation

$$
\frac{\partial u}{\partial t} = \alpha\,\frac{\partial^2 u}{\partial x^2}
$$

where:

- $u(x, t)$ — the field variable (e.g. temperature)
- $\alpha > 0$ — thermal diffusivity; larger $\alpha$ means faster diffusion

The Laplacian $\partial_{xx} u$ drives $u$ toward the spatial average: regions hotter
than their neighbours cool down, regions cooler than their neighbours warm up.

## Problem Setup

| Component | Expression |
|---|---|
| Domain | $x \in [0,1],\; t \in [0,1]$ |
| Diffusivity | $\alpha = 0.1$ |
| Left BC | $u(0, t) = 0$ |
| Right BC | $u(1, t) = 0$ |
| Initial condition | $u(x, 0) = \sin(\pi x)$ |

## Exact Solution

With zero Dirichlet boundary conditions and a sinusoidal initial condition the
solution is a **decaying standing wave**:

$$
u(x, t) = \sin(\pi x)\,e^{-\alpha \pi^2 t}
$$

The spatial mode $\sin(\pi x)$ is preserved exactly; only its amplitude decays
exponentially at rate $\alpha \pi^2$.

## Comparison with the Other PINN Notebooks

| Property | Heat | Wave | Burgers' |
|---|---|---|---|
| Order in time | 1st | 2nd | 1st |
| Linearity | Linear | Linear | **Nonlinear** |
| Initial conditions | 1 (displacement) | 2 (displacement + velocity) | 1 |
| PDE terms | $u_t,\; u_{xx}$ | $u_{tt},\; u_{xx}$ | $u_t,\; u\,u_x,\; u_{xx}$ |
| Loss terms | 2 (data + PDE) | **3** (data + PDE + vel. IC) | 2 (data + PDE) |

The heat equation is the **simplest** of the three: linear, first-order in time, and
requires only a standard two-term PINN loss. It is also the target equation of the
Cole–Hopf transformation that linearises Burgers' equation.

## Common Applications

- Heat conduction and thermal analysis
- Mass diffusion and mixing
- Image smoothing (Gaussian blurring solves the heat equation)
- Finance: the Black–Scholes equation reduces to the heat equation after a change of variables

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from tqdm import tqdm

In [ ]:
class HeatNet(nn.Module):
    """Fully-connected network approximating u(x, t) for the heat equation.

    Architecture: 2 → 20 → 30 → 30 → 20 → 20 → 1, Tanh activations throughout.
    Because the exact solution sin(πx)·exp(−αt) is smooth and monotonically
    decaying, this network is no wider than the BurgersNet baseline.
    """

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 20),  nn.Tanh(),
            nn.Linear(20, 30), nn.Tanh(),
            nn.Linear(30, 30), nn.Tanh(),
            nn.Linear(30, 20), nn.Tanh(),
            nn.Linear(20, 20), nn.Tanh(),
            nn.Linear(20, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
class HeatPINN:
    """Physics-Informed Neural Network solver for the 1D heat equation.

    PDE:  u_t = α·u_xx,  x ∈ [0, 1],  t ∈ [0, 1]
    BCs:  u(0, t) = u(1, t) = 0
    IC:   u(x, 0) = sin(πx)

    Exact solution:  u(x, t) = sin(πx)·exp(−α·π²·t)

    The heat equation is linear and first-order in time, so only one initial
    condition is needed. The loss has two components:
      loss_data — MSE on BC + IC labels.
      loss_pde  — MSE of the PDE residual u_t − α·u_xx at collocation points.
    """

    ALPHA: float = 0.1  # thermal diffusivity

    def __init__(self, h: float = 0.05, k: float = 0.05):
        """
        Args:
            h: Spatial grid spacing for collocation points.
            k: Temporal grid spacing for collocation points.
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = HeatNet().to(self.device)
        self.criterion = nn.MSELoss()
        self.iter = 0

        x = torch.linspace(0.0, 1.0, round(1.0 / h) + 1)
        t = torch.linspace(0.0, 1.0, round(1.0 / k) + 1)

        # --- Collocation grid (PDE residual enforced at every node) ---
        X_grid, T_grid = torch.meshgrid(x, t, indexing="ij")
        self.X_col = torch.stack([X_grid.ravel(), T_grid.ravel()], dim=1).to(self.device)
        self.X_col.requires_grad_(True)

        # --- Labeled training data: BCs + IC ---
        self.X_train, self.y_train = self._build_training_data(x, t)
        self.X_train = self.X_train.to(self.device)
        self.y_train = self.y_train.to(self.device)

        # --- Optimizers ---
        self.optimizer_adam = torch.optim.Adam(self.model.parameters())
        self.optimizer_lbfgs = torch.optim.LBFGS(
            self.model.parameters(),
            max_iter=50_000,
            max_eval=50_000,
            tolerance_grad=1e-7,
            tolerance_change=float(np.finfo(float).eps),
            line_search_fn="strong_wolfe",
            history_size=50,
        )

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------

    def _build_training_data(
        self, x: torch.Tensor, t: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Assemble labeled (X, y) pairs for BCs and IC."""

        # Left BC: u(0, t) = 0
        X_l, T_l = torch.meshgrid(x[:1], t, indexing="ij")
        bc_left = torch.stack([X_l.ravel(), T_l.ravel()], dim=1)
        y_left  = torch.zeros(len(bc_left), 1)

        # Right BC: u(1, t) = 0
        X_r, T_r = torch.meshgrid(x[-1:], t, indexing="ij")
        bc_right = torch.stack([X_r.ravel(), T_r.ravel()], dim=1)
        y_right  = torch.zeros(len(bc_right), 1)

        # IC: u(x, 0) = sin(πx)
        X_ic, T_ic = torch.meshgrid(x, t[:1], indexing="ij")
        ic   = torch.stack([X_ic.ravel(), T_ic.ravel()], dim=1)
        y_ic = torch.sin(torch.pi * ic[:, 0]).unsqueeze(1)

        return torch.cat([bc_left, bc_right, ic]), torch.cat([y_left, y_right, y_ic])

    def _loss(self) -> torch.Tensor:
        """Compute total loss = data loss + PDE residual loss.

        Serves as the closure for both Adam and L-BFGS. Zeroes gradients,
        performs forward passes, computes both loss components, runs
        backpropagation, and returns the scalar loss.
        """
        self.optimizer_adam.zero_grad()
        self.optimizer_lbfgs.zero_grad()

        # --- Data loss: BCs + IC ---
        y_pred    = self.model(self.X_train)
        loss_data = self.criterion(y_pred, self.y_train)

        # --- PDE residual: u_t − α·u_xx = 0 ---
        u = self.model(self.X_col)  # (N_col, 1)

        # First derivatives: ∂u/∂x and ∂u/∂t
        grad_1 = torch.autograd.grad(
            u, self.X_col,
            grad_outputs=torch.ones_like(u),
            create_graph=True, retain_graph=True,
        )[0]
        u_x = grad_1[:, 0]  # ∂u/∂x
        u_t = grad_1[:, 1]  # ∂u/∂t

        # Second spatial derivative: ∂²u/∂x²
        grad_2 = torch.autograd.grad(
            u_x, self.X_col,
            grad_outputs=torch.ones_like(u_x),
            create_graph=True, retain_graph=True,
        )[0]
        u_xx = grad_2[:, 0]  # ∂²u/∂x²

        # Heat equation residual: u_t − α·u_xx = 0
        residual = u_t - self.ALPHA * u_xx
        loss_pde = self.criterion(residual, torch.zeros_like(residual))

        loss = loss_data + loss_pde
        loss.backward()

        self.iter += 1
        if self.iter % 100 == 0:
            print(f"  iter {self.iter:>6d}  loss {loss.item():.6e}")

        return loss

    # ------------------------------------------------------------------
    # Public interface
    # ------------------------------------------------------------------

    def train(self, adam_steps: int = 1000) -> None:
        """Two-phase training: Adam pre-conditioning then L-BFGS fine-tuning."""
        self.model.train()

        print("Phase 1 — Adam optimizer")
        for _ in tqdm(range(adam_steps), desc="Adam"):
            self._loss()
            self.optimizer_adam.step()

        print("\nPhase 2 — L-BFGS optimizer")
        self.optimizer_lbfgs.step(self._loss)
        print("Done.")

    def predict(self, X: torch.Tensor) -> np.ndarray:
        """Evaluate the trained network on a set of (x, t) points.

        Args:
            X: Tensor of shape (N, 2) with columns [x, t].

        Returns:
            NumPy array of shape (N, 1) with predicted u values.
        """
        self.model.eval()
        with torch.no_grad():
            return self.model(X.to(self.device)).cpu().numpy()

In [ ]:
pinn = HeatPINN(h=0.05, k=0.05)
pinn.train(adam_steps=1000)

In [ ]:
# --- Evaluate PINN and exact solution on a fine grid ---
nx, nt = 300, 150
x_plot = torch.linspace(0.0, 1.0, nx)
t_plot = torch.linspace(0.0, 1.0, nt)

X_grid, T_grid = torch.meshgrid(x_plot, t_plot, indexing="ij")
X_eval = torch.stack([X_grid.ravel(), T_grid.ravel()], dim=1)

alpha  = HeatPINN.ALPHA
u_pred  = pinn.predict(X_eval).reshape(nx, nt)
u_exact = (torch.sin(torch.pi * X_grid) * torch.exp(-alpha * torch.pi**2 * T_grid)).numpy()
abs_err = np.abs(u_pred - u_exact)

# --- Space-time heatmaps: prediction | exact solution | absolute error ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].contourf(t_plot, x_plot, u_pred,  levels=100, cmap="hot_r")
fig.colorbar(im0, ax=axes[0], label="u(x, t)")
axes[0].set_xlabel("t"); axes[0].set_ylabel("x")
axes[0].set_title("PINN Prediction")

im1 = axes[1].contourf(t_plot, x_plot, u_exact, levels=100, cmap="hot_r")
fig.colorbar(im1, ax=axes[1], label="u(x, t)")
axes[1].set_xlabel("t"); axes[1].set_ylabel("x")
axes[1].set_title(r"Exact: $\sin(\pi x)\,e^{-\alpha\pi^2 t}$")

im2 = axes[2].contourf(t_plot, x_plot, abs_err, levels=100, cmap="Blues")
fig.colorbar(im2, ax=axes[2], label="|error|")
axes[2].set_xlabel("t"); axes[2].set_ylabel("x")
axes[2].set_title(f"Absolute Error  (max = {abs_err.max():.2e})")

plt.tight_layout()
plt.show()

# --- Snapshot comparison at selected time slices (shows exponential decay) ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, t_snap in zip(axes, [0.0, 0.5, 1.0]):
    idx = int(t_snap * (nt - 1))
    ax.plot(x_plot, u_exact[:, idx], "k--", label="Exact", linewidth=2)
    ax.plot(x_plot, u_pred[:, idx],  "r-",  label="PINN",  linewidth=1.5)
    ax.set_xlabel("x")
    ax.set_ylabel("u(x, t)")
    ax.set_title(f"t = {t_snap:.1f}  (amp ≈ {np.exp(-alpha * np.pi**2 * t_snap):.3f})")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Heat equation — exponential amplitude decay over time", y=1.02)
plt.tight_layout()
plt.show()